In [5]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week10-lesson-2"). \
config("spark.sql.warehouse.dir", f"/user/itv027484/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [8]:
order_schema= 'order_id long, order_date string , customer_id long, order_status string'

In [9]:
orders_df = spark.read.format('csv').schema(order_schema).load('/public/trendytech/retail_db/ordersnew')

### Partition skew - check the spark UI after execution

In [10]:
orders_df.rdd.getNumPartitions()

23

In [5]:
orders_df.show(3)

+--------+--------------------+-----------+------------+
|order_id|          order_date|customer_id|order_status|
+--------+--------------------+-----------+------------+
|    2480|2013-08-07 00:00:...|       3807|    COMPLETE|
|   30479|2014-01-30 00:00:...|       9265|    COMPLETE|
|    2481|2013-08-07 00:00:...|       2476|    COMPLETE|
+--------+--------------------+-----------+------------+
only showing top 3 rows



In [22]:
orders_df.groupBy("order_status").count().collect()

[Row(order_status='PENDING_PAYMENT', count=5636250),
 Row(order_status='COMPLETE', count=46008801),
 Row(order_status='ON_HOLD', count=1424250),
 Row(order_status='PAYMENT_REVIEW', count=273375),
 Row(order_status='PROCESSING', count=3103125),
 Row(order_status='CLOSED', count=2833500),
 Row(order_status='SUSPECTED_FRAUD', count=584250),
 Row(order_status='PENDING', count=2853750),
 Row(order_status='CANCELED', count=535500)]

In [1]:
## after the shuffle, only one of the partition becomes much larger than others due to the data skew. This slows down the job

#### COMPLETE status occurs much more than other statuses

In [7]:
mapping_schema = "status string, code int"

In [8]:
mapping_df = spark.read.format('csv').option("delimiter","|").schema(mapping_schema).load("/public/trendytech/datasets/mapping_data")

In [9]:
mapping_df.show()

+---------------+----+
|         status|code|
+---------------+----+
|PENDING_PAYMENT|   1|
|       COMPLETE|   2|
|        ON_HOLD|   3|
| PAYMENT_REVIEW|   4|
|     PROCESSING|   5|
|         CLOSED|   6|
|SUSPECTED_FRAUD|   7|
|        PENDING|   8|
|       CANCELED|   9|
+---------------+----+



In [10]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold','-1')

In [11]:
joined_df = orders_df.join(mapping_df,orders_df.order_status == mapping_df.status,"inner").write.format('noop').mode('overwrite').save()

In [4]:
## broad cast join can signficantly help in data skew, as it tries to avoid shuffling by broadcasting one DF to all partitions

In [12]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold','10485760b')

In [13]:
joined_df = orders_df.join(mapping_df,orders_df.order_status == mapping_df.status,"inner").write.format('noop').mode('overwrite').save()

## 

In [16]:
cust_new = spark.read.format('csv').load('/public/trendytech/retail_db/customersnew')

In [21]:
cust_new.rdd.getNumPartitions()

2

In [17]:
cust_dfnew = cust_new.toDF("cust_id","cust_fname","cust_lname","cust_email","cust_pass","cust_addr1","cust_city","cust_state","cust_zip")

In [18]:
cust_dfnew.show(2)

+-------+----------+----------+----------+---------+--------------------+-----------+----------+--------+
|cust_id|cust_fname|cust_lname|cust_email|cust_pass|          cust_addr1|  cust_city|cust_state|cust_zip|
+-------+----------+----------+----------+---------+--------------------+-----------+----------+--------+
|      1|   Richard| Hernandez| XXXXXXXXX|XXXXXXXXX|  6303 Heather Plaza|Brownsville|        TX|   78521|
|      2|      Mary|   Barrett| XXXXXXXXX|XXXXXXXXX|9526 Noble Embers...|  Littleton|        CO|   80126|
+-------+----------+----------+----------+---------+--------------------+-----------+----------+--------+
only showing top 2 rows



In [19]:
joineddf1 = orders_df.join(cust_dfnew.distinct(),orders_df.customer_id == cust_dfnew.distinct().cust_id,"inner").write.format('noop').mode('overwrite').save()

### even though after the distinct, the data has reduced to < 10 MB, Broadcast join is not invoked. 

## This requires AQE

In [2]:
# 3 common problem use cases
# 1. Wide transformations that lead to 200 shuffle partitions but only few of the partitions have data.
# 2. A dominating key leading to Partition Skew.
# 3. Joining dataframes on distinct keys of one of the dataframe

In [3]:
# Adaptive Query Execution (AQE)
# Is a feature that is available from Spark Major Version 3.0 onwards. This
# feature provides the following benefits to improve query performance.
# 1. Dynamically Coalescing the number of shuffle partitions
# 2. Dynamically handling Partition Skew
# 3. Dynamically Switching Join Strategies


In [23]:
spark.conf.get("spark.sql.adaptive.enabled")

'false'

In [19]:
spark.conf.set("spark.sql.adaptive.enabled",True)

### only few partitions will be created after shuffle due to AQE

In [ ]:
orders_df.groupBy("order_status").count().collect()

In [21]:
spark.conf.set("spark.sql.adaptive.enabled",False)

### 200 partitions after shuffle by default if NO AQE - even when lot of partitions are empty

In [ ]:
orders_df.groupBy("order_status").count().collect()